# Homologação dirigida de otimização

Este notebook existe para **fechar os poucos pontos ainda não comprovados** antes da documentação final ser entregue ao desenvolvedor.

Ele não substitui o laboratório amplo já executado.

## Objetivos

1. Homologar Q4 em **múltiplos clientes e múltiplas datas**:
   - baseline funcional;
   - busca regressiva por competência exata;
   - equivalência de conteúdo;
   - número de probes;
   - tempo total.

2. Registrar fotografia atual da Q5:
   - contexto;
   - janela oficial;
   - menor/maior `DT_TRAN`;
   - menor/maior identificador técnico;
   - `MAX(TS_ATL_TRAN)` quando disponível;
   - comparar com uma referência anterior opcional fornecida pelo desenvolvedor.

3. Inspecionar estaticamente o notebook/código real do projeto:
   - `count`;
   - `collect`;
   - `first`;
   - `show`;
   - `persist/cache`;
   - `groupBy`;
   - `orderBy`;
   - `join`;
   - `Window`;
   - `createOrReplaceTempView`;
   - `get_from_spark`;
   - células com maior concentração de actions.

4. Produzir um JSON único de homologação para alimentar a documentação técnica.

## Segurança

- read-only;
- nenhuma escrita em fonte;
- nenhuma tabela permanente;
- nenhuma query deliberadamente não-sargable;
- nenhuma variante `ALTO/EXTREMO`;
- falhas individuais são registradas e não interrompem as demais;
- nomes físicos de fontes não existem neste arquivo: usar somente `{schema}.{nome_tabela}` via configuração.

## Esta versão

Esta cópia está **preenchida para o smoke test do caso de referência**.
Execute de cima para baixo. Esta cópia está totalmente preenchida para o caso de referência.

Não é necessário preencher fontes nem variáveis de ambiente nesta cópia de execução.


## 1. Sessão Spark corporativa

In [ ]:
from traceback import format_exc

try:
    if globals().get("spark") is None:
        try:
            from src.utils.gerenciador_local_v2 import GerenciadorLocal
        except Exception:
            from gerenciador_local_v2 import GerenciadorLocal

        gerenciador_local = GerenciadorLocal(
            nome_sessao="homologacao-otimizacao-dirigida",
            exibir_configuracao=False,
            ativar_logs=True,
        )
        spark = gerenciador_local.criar_sessao_spark(db2=True)
        print("[HOMOLOG] Sessão Spark criada.")
    else:
        print("[HOMOLOG] Reutilizando sessão Spark.")
except Exception as exc:
    print(type(exc).__name__, str(exc))
    print(format_exc())
    raise

## 2. Utilitários corporativos

In [ ]:
from IPython import get_ipython

def carregar(candidatos):
    ip = get_ipython()
    erros = []
    for caminho in candidatos:
        try:
            ip.run_line_magic("run", caminho)
            print("[HOMOLOG] carregado:", caminho)
            return caminho
        except Exception as exc:
            erros.append(f"{caminho}: {type(exc).__name__}: {exc}")
    raise RuntimeError(" | ".join(erros))

carregar([
    "./src/utils/gerenciador_spark_v2.ipynb",
    "./gerenciador_spark_v2.ipynb",
])
carregar([
    "./src/utils/gerenciador_db2_spark_v2.ipynb",
    "./gerenciador_db2_spark_v2.ipynb",
])

## 3. Parâmetros — já preenchidos

Esta cópia está pronta para o smoke test controlado.

Já estão definidos:

- cliente de referência;
- data de execução;
- fonte Q4;
- fonte Q5;
- janela Q5;
- referência anterior `83 / 56`;
- limites de timeout/repetição.

**Não é necessário digitar nada. Execute `Run All`.**


In [ ]:
%%spark

# ============================================================
# HOMOLOGAÇÃO DIRIGIDA — PRONTO PARA RODAR
# ============================================================

# Caso de referência
CD_CLIS = [
    468459778,
]

DATAS_EXECUCAO_TESTE = [
    "2026-09-01",
]

# Fontes do processo em homologação.
# Esta cópia é específica para a execução controlada atual.
CONFIG_HOMOLOG_FONTES = {
    "Q4": {
        "schema": "DB2D1D",
        "nome_tabela": "DVS_GRDR_FNCO_PF",
    },
    "Q5": {
        "schema": "DB2GFP",
        "nome_tabela": "TRAN_RLZD_INST_PCT",
    },
}

# Fotografia anterior — somente referência comparativa.
REFERENCIA_Q5 = {
    "468459778": {
        "dt_ref_ini": "2026-07-20",
        "dt_ref_fim": "2026-08-19",
        "contexto": 83,
        "oficial": 56,
    }
}

FETCHSIZE = 10_000
QUERY_TIMEOUT = 120
MAX_MESES_BUSCA_Q4 = 60
REPETICOES_Q4 = 2
DIAS_CONTEXTO_Q5 = 5

print("[HOMOLOG] Parâmetros preenchidos.")
print("[HOMOLOG] Cliente:", CD_CLIS)
print("[HOMOLOG] Data:", DATAS_EXECUCAO_TESTE)


In [ ]:
# Opcional. O smoke test Q4/Q5 roda sem este item.
# Preencha depois apenas se quiser incluir o scan estático do código real.
CAMINHOS_CODIGO_PROJETO = []


## 4. Framework remoto resiliente

In [ ]:
%%spark

import calendar
import datetime
import hashlib
import json
import os
import statistics
import time
import traceback
from datetime import timedelta
from decimal import Decimal

from pyspark.sql import functions as F

# Estado sempre definido para que um preflight bloqueado não gere NameError em cascata.
HOMOLOG_READY = False
HOMOLOG_CONFIG_ERRO = None
FONTE_Q4 = None
FONTE_Q5 = None
conector_db2 = None

RESULTADOS = []
Q4_DETALHE = []
Q5_DETALHE = []

def parse_data(v):
    if isinstance(v, datetime.date):
        return v
    return datetime.date.fromisoformat(str(v)[:10])

DATAS_TESTE = [parse_data(x) for x in DATAS_EXECUCAO_TESTE] if DATAS_EXECUCAO_TESTE else []

def _fonte_completa(cfg, chave):
    if not isinstance(cfg, dict):
        raise TypeError(f"{chave}: configuração inválida.")
    schema = str(cfg.get("schema", "")).strip()
    tabela = str(cfg.get("nome_tabela", "")).strip()

    if not schema:
        raise ValueError(f"{chave}: schema vazio.")
    if not tabela:
        raise ValueError(f"{chave}: nome_tabela vazio.")
    if "{" in schema or "}" in schema or "{" in tabela or "}" in tabela:
        raise ValueError(f"{chave}: placeholder não substituído.")

    return f"{schema}.{tabela}"

def reg(grupo, teste, status="OK", inicio=None, detalhe=None, **kwargs):
    item = {
        "grupo": grupo,
        "teste": teste,
        "status": status,
        "tempo_s": None if inicio is None else time.perf_counter() - inicio,
        "detalhe": detalhe,
        **kwargs,
    }
    RESULTADOS.append(item)
    print(
        f'[HOMOLOG][{status}] {grupo} :: {teste}'
        + ("" if item["tempo_s"] is None else f' | {item["tempo_s"]:.3f}s')
    )
    if detalhe:
        print("   ", str(detalhe)[:1000])
    return item

def norm(v):
    if isinstance(v, Decimal):
        return str(v)
    if isinstance(v, (datetime.date, datetime.datetime)):
        return v.isoformat()
    return v

def row_dict(row):
    if row is None:
        return None
    return {k: norm(v) for k, v in row.asDict(recursive=True).items()}

def hash_dict(d):
    if d is None:
        return None
    payload = json.dumps(d, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def mes_anterior_primeiro_dia(d):
    total = d.year * 12 + d.month - 2
    ano, mes0 = divmod(total, 12)
    return datetime.date(ano, mes0 + 1, 1)

try:
    if "CONFIG_HOMOLOG_FONTES" not in globals():
        raise RuntimeError(
            "CONFIG_HOMOLOG_FONTES ausente no Spark. "
            "Execute primeiro a célula local 'Fontes — entrada em runtime'."
        )

    FONTE_Q4 = _fonte_completa(CONFIG_HOMOLOG_FONTES.get("Q4"), "Q4")
    FONTE_Q5 = _fonte_completa(CONFIG_HOMOLOG_FONTES.get("Q5"), "Q5")

    if not CD_CLIS:
        raise RuntimeError("CD_CLIS está vazio.")

    for valor in CD_CLIS:
        if isinstance(valor, bool) or not str(valor).lstrip("+-").isdigit():
            raise TypeError(f"CD_CLI inválido: {valor!r}")

    CD_CLIS = [int(x) for x in CD_CLIS]

    if not DATAS_TESTE:
        raise RuntimeError("DATAS_EXECUCAO_TESTE está vazio.")

    conector_db2 = criar_conector_db2_spark(env=dict(os.environ))

    HOMOLOG_READY = True
    print("[HOMOLOG][PREFLIGHT_OK] configuração recebida e validada.")
    print("[HOMOLOG] Casos:", len(CD_CLIS), "cliente(s) x", len(DATAS_TESTE), "data(s)")

except Exception as exc:
    HOMOLOG_CONFIG_ERRO = f"{type(exc).__name__}: {exc}"
    reg(
        "PREFLIGHT",
        "CONFIGURACAO",
        status="BLOQUEADO",
        detalhe=HOMOLOG_CONFIG_ERRO,
    )

    # Zera somente coleções de execução para que Run All continue sem disparar consultas
    # e sem gerar NameError nas células posteriores.
    CD_CLIS = []
    DATAS_TESTE = []
    REFERENCIA_Q5 = {}

    print("[HOMOLOG] Nenhuma consulta será executada enquanto o preflight estiver bloqueado.")


## 5. Q4 — baseline funcional

A baseline é a semântica que queremos preservar:

```text
maior DT_REF <= data de execução
```

com desempate por códigos quando necessário.

In [ ]:
%%spark

def q4_baseline(cd_cli, data_execucao):
    sql = f'''
    SELECT
        CD_CLI,
        DT_REF,
        CD_MAC_PRFL_CLI,
        NM_MAC_PRFL_CLI,
        CD_MIC_PRFL_CLI,
        NM_MIC_PRFL_CLI
    FROM {FONTE_Q4}
    WHERE CD_CLI = {cd_cli}
      AND DT_REF <= DATE('{data_execucao.isoformat()}')
    ORDER BY
        DT_REF DESC,
        CD_MAC_PRFL_CLI DESC,
        CD_MIC_PRFL_CLI DESC
    FETCH FIRST 1 ROW ONLY
    '''
    return conector_db2.sql(
        sql,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT,
    ).first()

## 6. Q4 — candidata: busca regressiva por competência exata

A candidata explora consultas pontuais por:

```text
DT_REF exata + CD_CLI
```

Começa no mês da execução e recua até encontrar a primeira linha.

Ela **não é promovida automaticamente**. O objetivo desta bateria é provar se devolve o mesmo resultado da baseline para toda a amostra.

In [ ]:
%%spark

def q4_regressiva(cd_cli, data_execucao, max_meses=MAX_MESES_BUSCA_Q4):
    competencia = data_execucao.replace(day=1)
    probes = []

    for i in range(max_meses):
        inicio_probe = time.perf_counter()
        sql = f'''
        SELECT
            CD_CLI,
            DT_REF,
            CD_MAC_PRFL_CLI,
            NM_MAC_PRFL_CLI,
            CD_MIC_PRFL_CLI,
            NM_MIC_PRFL_CLI
        FROM {FONTE_Q4}
        WHERE DT_REF = DATE('{competencia.isoformat()}')
          AND CD_CLI = {cd_cli}
        ORDER BY
            CD_MAC_PRFL_CLI DESC,
            CD_MIC_PRFL_CLI DESC
        FETCH FIRST 1 ROW ONLY
        '''
        linha = conector_db2.sql(
            sql,
            fetchsize=100,
            query_timeout=min(60, QUERY_TIMEOUT),
        ).first()

        probes.append({
            "competencia": competencia.isoformat(),
            "tempo_s": time.perf_counter() - inicio_probe,
            "encontrou": linha is not None,
        })

        if linha is not None:
            return linha, probes

        competencia = mes_anterior_primeiro_dia(competencia)

    return None, probes

## 7. Q4 — execução multi-cliente/multi-data

In [ ]:
%%spark

Q4_RESUMO = []

for cd_cli in CD_CLIS:
    for data_execucao in DATAS_TESTE:
        # Baseline.
        inicio = time.perf_counter()
        try:
            linha_base = q4_baseline(cd_cli, data_execucao)
            tempo_base = time.perf_counter() - inicio
            base = row_dict(linha_base)
            status_base = "OK"
        except Exception as exc:
            tempo_base = time.perf_counter() - inicio
            base = None
            status_base = "ERRO"
            reg(
                "Q4",
                f"BASE_{cd_cli}_{data_execucao}",
                "ERRO",
                detalhe=f"{type(exc).__name__}: {exc}",
            )

        # Candidata.
        inicio = time.perf_counter()
        try:
            linha_cand, probes = q4_regressiva(cd_cli, data_execucao)
            tempo_cand = time.perf_counter() - inicio
            cand = row_dict(linha_cand)
            status_cand = "OK"
        except Exception as exc:
            tempo_cand = time.perf_counter() - inicio
            cand = None
            probes = []
            status_cand = "ERRO"
            reg(
                "Q4",
                f"CAND_{cd_cli}_{data_execucao}",
                "ERRO",
                detalhe=f"{type(exc).__name__}: {exc}",
            )

        equivalentes = (
            status_base == "OK"
            and status_cand == "OK"
            and base == cand
        )

        item = {
            "cd_cli": cd_cli,
            "data_execucao": data_execucao.isoformat(),
            "baseline": base,
            "candidata": cand,
            "hash_baseline": hash_dict(base),
            "hash_candidata": hash_dict(cand),
            "equivalente": equivalentes,
            "tempo_baseline_s": tempo_base,
            "tempo_candidata_s": tempo_cand,
            "speedup": (
                None
                if tempo_cand <= 0
                else tempo_base / tempo_cand
            ),
            "probes": len(probes),
            "probes_detalhe": probes,
        }
        Q4_RESUMO.append(item)
        Q4_DETALHE.append(item)

        reg(
            "Q4",
            f"{cd_cli}_{data_execucao}",
            "OK" if equivalentes else "DIVERGENTE",
            equivalente=equivalentes,
            tempo_baseline_s=tempo_base,
            tempo_candidata_s=tempo_cand,
            probes=len(probes),
            detalhe={
                "baseline": base,
                "candidata": cand,
            },
        )

## 8. Q4 — repetição dos casos equivalentes para estabilidade

In [ ]:
%%spark

Q4_REPETICOES = []

for item in Q4_RESUMO:
    if not item["equivalente"]:
        continue

    cd_cli = item["cd_cli"]
    data_execucao = parse_data(item["data_execucao"])

    for rep in range(1, REPETICOES_Q4 + 1):
        inicio = time.perf_counter()
        try:
            _, probes = q4_regressiva(cd_cli, data_execucao)
            dur = time.perf_counter() - inicio
            Q4_REPETICOES.append({
                "cd_cli": cd_cli,
                "data_execucao": data_execucao.isoformat(),
                "repeticao": rep,
                "tempo_s": dur,
                "probes": len(probes),
                "status": "OK",
            })
        except Exception as exc:
            Q4_REPETICOES.append({
                "cd_cli": cd_cli,
                "data_execucao": data_execucao.isoformat(),
                "repeticao": rep,
                "tempo_s": time.perf_counter() - inicio,
                "probes": None,
                "status": "ERRO",
                "erro": f"{type(exc).__name__}: {exc}",
            })

## 9. Q5 — fotografia atual

Esta etapa **não tenta “corrigir” uma diferença histórica**.

Ela apenas captura evidência suficiente para decidir se uma mudança entre execuções é compatível com atualização/late arrival:

- linhas no halo;
- linhas na janela oficial;
- menor/maior data econômica;
- menor/maior identificador técnico;
- maior timestamp de atualização/inclusão, quando a fonte expuser as colunas.

In [ ]:
%%spark

def q5_fotografia(cd_cli, dt_ref_ini, dt_ref_fim):
    dt_ctx_ini = dt_ref_ini - timedelta(days=DIAS_CONTEXTO_Q5)
    dt_ctx_fim = dt_ref_fim + timedelta(days=DIAS_CONTEXTO_Q5)

    sql = f'''
    SELECT
        NR_TRAN_INST_PCT,
        CD_CLI,
        DT_TRAN,
        TS_INCL_TRAN,
        TS_ATL_TRAN
    FROM {FONTE_Q5}
    WHERE CD_CLI = {cd_cli}
      AND DT_TRAN >= DATE('{dt_ctx_ini.isoformat()}')
      AND DT_TRAN <= DATE('{dt_ctx_fim.isoformat()}')
      AND CD_EST_TRAN_INST = 0
    '''

    df = conector_db2.sql(
        sql,
        fetchsize=FETCHSIZE,
        query_timeout=QUERY_TIMEOUT,
    )

    oficial = (
        (F.col("DT_TRAN") >= F.lit(dt_ref_ini))
        & (F.col("DT_TRAN") <= F.lit(dt_ref_fim))
    )

    r = df.agg(
        F.count(F.lit(1)).cast("long").alias("QT_CONTEXTO"),
        F.sum(F.when(oficial, 1).otherwise(0)).cast("long").alias("QT_OFICIAL"),
        F.min("DT_TRAN").alias("DT_MIN"),
        F.max("DT_TRAN").alias("DT_MAX"),
        F.min("NR_TRAN_INST_PCT").alias("ID_MIN"),
        F.max("NR_TRAN_INST_PCT").alias("ID_MAX"),
        F.max("TS_INCL_TRAN").alias("TS_INCL_MAX"),
        F.max("TS_ATL_TRAN").alias("TS_ATL_MAX"),
    ).first()

    return {
        "cd_cli": cd_cli,
        "dt_ref_ini": dt_ref_ini.isoformat(),
        "dt_ref_fim": dt_ref_fim.isoformat(),
        "dt_contexto_ini": dt_ctx_ini.isoformat(),
        "dt_contexto_fim": dt_ctx_fim.isoformat(),
        **row_dict(r),
    }

## 10. Q5 — casos configurados

Para que esta etapa rode, forneça em `REFERENCIA_Q5` ao menos:

```text
dt_ref_ini
dt_ref_fim
```

`contexto` e `oficial` anteriores são opcionais.

Assim o notebook pode ser usado por qualquer desenvolvedor e não pressupõe uma janela fixa.

In [ ]:
%%spark

Q5_RESUMO = []

for chave, ref in REFERENCIA_Q5.items():
    try:
        cd_cli = int(chave)
    except Exception:
        cd_cli = int(ref.get("cd_cli"))

    dt_ref_ini = parse_data(ref["dt_ref_ini"])
    dt_ref_fim = parse_data(ref["dt_ref_fim"])

    inicio = time.perf_counter()
    try:
        atual = q5_fotografia(cd_cli, dt_ref_ini, dt_ref_fim)
        anterior_contexto = ref.get("contexto")
        anterior_oficial = ref.get("oficial")

        atual["anterior_contexto"] = anterior_contexto
        atual["anterior_oficial"] = anterior_oficial
        atual["delta_contexto"] = (
            None
            if anterior_contexto is None
            else int(atual["QT_CONTEXTO"]) - int(anterior_contexto)
        )
        atual["delta_oficial"] = (
            None
            if anterior_oficial is None
            else int(atual["QT_OFICIAL"]) - int(anterior_oficial)
        )

        Q5_RESUMO.append(atual)
        Q5_DETALHE.append(atual)
        reg(
            "Q5",
            f"FOTOGRAFIA_{cd_cli}",
            "OK",
            inicio=inicio,
            detalhe=atual,
        )
    except Exception as exc:
        reg(
            "Q5",
            f"FOTOGRAFIA_{cd_cli}",
            "ERRO",
            inicio=inicio,
            detalhe=f"{type(exc).__name__}: {exc}\n{traceback.format_exc()}",
        )

## 11. Consolidação remota

In [ ]:
%%spark

q4_validos = [x for x in Q4_RESUMO if x.get("equivalente") is not None]
q4_equiv = [x for x in q4_validos if x["equivalente"]]
q4_div = [x for x in q4_validos if not x["equivalente"]]

tempos_speedup = [
    x["speedup"]
    for x in q4_equiv
    if x.get("speedup") is not None
]

RESUMO_REMOTO = {
    "preflight": {
        "ready": HOMOLOG_READY,
        "erro": HOMOLOG_CONFIG_ERRO,
    },
    "q4": {
        "casos_total": len(Q4_RESUMO),
        "equivalentes": len(q4_equiv),
        "divergentes": len(q4_div),
        "equivalencia_pct": (
            None
            if not Q4_RESUMO
            else 100.0 * len(q4_equiv) / len(Q4_RESUMO)
        ),
        "speedup_mediana": (
            None
            if not tempos_speedup
            else statistics.median(tempos_speedup)
        ),
        "max_probes": (
            None
            if not Q4_RESUMO
            else max(x["probes"] for x in Q4_RESUMO)
        ),
    },
    "q5": {
        "casos": len(Q5_RESUMO),
        "fotografias": Q5_RESUMO,
    },
}

print(json.dumps(RESUMO_REMOTO, ensure_ascii=False, indent=2, default=str))

RELATORIO_REMOTO_JSON = json.dumps(
    {
        "resumo": RESUMO_REMOTO,
        "q4_detalhe": Q4_DETALHE,
        "q4_repeticoes": Q4_REPETICOES,
        "q5_detalhe": Q5_DETALHE,
        "resultados": RESULTADOS,
    },
    ensure_ascii=False,
    default=str,
)

## 12. Scan estático do código real do projeto

In [ ]:
import json
import re
from pathlib import Path

PADROES = {
    "count": r"\.count\s*\(",
    "collect": r"\.collect\s*\(",
    "first": r"\.first\s*\(",
    "take": r"\.take\s*\(",
    "show": r"\.show\s*\(",
    "persist": r"\.persist\s*\(",
    "cache": r"\.cache\s*\(",
    "groupBy": r"\.groupBy\s*\(",
    "orderBy": r"\.orderBy\s*\(",
    "join": r"\.join\s*\(",
    "Window": r"\bWindow\b",
    "tempView": r"createOrReplaceTempView",
    "get_from_spark": r"get_from_spark",
}

ANALISE_ESTATICA = []

def analisar_texto(nome, blocos):
    totais = {k: 0 for k in PADROES}
    ranking = []

    for idx, src in blocos:
        cont = {k: len(re.findall(rx, src)) for k, rx in PADROES.items()}
        score_actions = sum(
            cont[k] for k in ["count","collect","first","take","show","get_from_spark"]
        )
        score_total = sum(cont.values())

        for k, v in cont.items():
            totais[k] += v

        if score_total:
            ranking.append({
                "bloco": idx,
                "score_actions": score_actions,
                "score_total": score_total,
                "contagens": cont,
                "inicio": src[:300].replace("\n", " "),
            })

    return {
        "arquivo": nome,
        "totais": totais,
        "top_blocos": sorted(
            ranking,
            key=lambda x: (x["score_actions"], x["score_total"]),
            reverse=True,
        )[:30],
    }

for caminho in CAMINHOS_CODIGO_PROJETO:
    p = Path(caminho)
    try:
        if p.suffix.lower() == ".ipynb":
            data = json.loads(p.read_text(encoding="utf-8"))
            blocos = [
                (i, "".join(c.get("source", [])))
                for i, c in enumerate(data.get("cells", []))
                if c.get("cell_type") == "code"
            ]
        else:
            txt = p.read_text(encoding="utf-8")
            blocos = [(1, txt)]

        resultado = analisar_texto(str(p), blocos)
        ANALISE_ESTATICA.append(resultado)

        print("\n" + "="*100)
        print(p)
        print("="*100)
        print("TOTAIS:", resultado["totais"])
        print("TOP BLOCOS:")
        for x in resultado["top_blocos"][:15]:
            print(x)
    except Exception as exc:
        ANALISE_ESTATICA.append({
            "arquivo": str(p),
            "erro": f"{type(exc).__name__}: {exc}",
        })
        print("[ERRO]", p, type(exc).__name__, exc)

## 13. Coleta e relatório local

In [ ]:
import json
from pathlib import Path

try:
    remoto = json.loads(spark.get_from_spark("RELATORIO_REMOTO_JSON"))
except Exception as exc:
    remoto = {
        "erro_coleta_remota": f"{type(exc).__name__}: {exc}"
    }

RELATORIO_FINAL = {
    **remoto,
    "analise_estatica": ANALISE_ESTATICA,
}

saida = Path.cwd() / "relatorio_homologacao_otimizacao.json"
saida.write_text(
    json.dumps(
        RELATORIO_FINAL,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Relatório salvo em:", saida)
if isinstance(remoto, dict):
    preflight = remoto.get("resumo", {}).get("preflight", {})
    print("Preflight remoto:", preflight)

# Critérios de decisão

## Q4 pode ser recomendada ao desenvolvedor quando

- nenhuma divergência funcional na amostra;
- amostra suficientemente diversa;
- comportamento estável em repetições;
- número de probes operacionalmente aceitável;
- ganho consistente em relação à baseline.

Não exigir `100%` apenas no sentido estatístico: **qualquer divergência funcional é bloqueante** até ser explicada.

## Q5

A diferença entre fotografias deve ser registrada como:

- atualização/late arrival comprovada;
- diferença de filtro;
- ou causa não determinada.

Não alterar regra funcional enquanto a causa não estiver explicada.

## Actions / lineage

O relatório estático não prova custo sozinho.

Ele serve para apontar ao desenvolvedor exatamente quais células/blocos precisam de instrumentação fina no projeto real.

## Saída para a documentação final

Depois da execução, envie:

```text
relatorio_homologacao_otimizacao.json
```

A partir desse arquivo será possível produzir o handoff final com:

- PROVADO;
- RECOMENDADO;
- CANDIDATO;
- NÃO ALTERAR;
- TESTE PENDENTE;
- ordem de implementação;
- critérios de aceite;
- riscos;
- evidências de performance.